# Lesson 4 — Self-attention

After lesson 3 every character is a vector, but each vector only knows about itself.
Here I build one **attention head** by hand so each token can look at the other tokens and pull in what it needs.

The formula: `attention(Q, K, V) = softmax(Q Kᵀ / √d_k) V`

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

## Rebuild the embedded batch (lessons 2–3)
Same recipe as before: character vocabulary, the whole text as integer ids, then a random batch of 4 chunks of 8 characters.

In [2]:
text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
block_size, batch_size, n_embd = 8, 4, 32

In [3]:
def get_batch(src, block, bs):
    ix = torch.randint(len(src) - block, (bs,))
    return (torch.stack([src[i:i + block] for i in ix]),
            torch.stack([src[i + 1:i + block + 1] for i in ix]))

Token embedding + position embedding gives `x`, shaped `(B, T, C)` = (batch, time, channels).

In [4]:
xb, yb = get_batch(data, block_size, batch_size)
x = nn.Embedding(vocab_size, n_embd)(xb) + nn.Embedding(block_size, n_embd)(torch.arange(block_size))
B, T, C = x.shape

## Three projections: query, key, value
Each token gets three smaller vectors from its embedding:
- **query** — what I'm looking for
- **key** — what I contain
- **value** — what I hand over if someone picks me

In [5]:
# --- one attention head, from scratch ---
head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

## Scores → weights → output
- `q @ kᵀ` compares every query with every key → a `(T, T)` score table per sequence.
- Multiply by `head_size ** -0.5` so the scores don't get huge (which would make softmax nearly one-hot).
- Softmax each row → weights that sum to 1.
- Weighted sum of values → the head's output for each token.

In [6]:
k, q, v = key(x), query(x), value(x)              # each (B, T, head_size)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5  # (B, T, T) scaled scores
wei = F.softmax(wei, dim=-1)                        # each row sums to 1
out = wei @ v                                       # (B, T, head_size)

In [7]:
print("q, k, v          :", tuple(q.shape))
print("attention weights:", tuple(wei.shape), "(query token x key token)")
print("every row sums to 1:", bool(torch.allclose(wei.sum(-1), torch.ones(B, T))))
print("head output      :", tuple(out.shape))
print("\nattention matrix for one sequence (row i = how token i spreads its attention):")
print(wei[0].round(decimals=2))

q, k, v          : (4, 8, 16)
attention weights: (4, 8, 8) (query token x key token)
every row sums to 1: True
head output      : (4, 8, 16)

attention matrix for one sequence (row i = how token i spreads its attention):
tensor([[0.1900, 0.1200, 0.0800, 0.2300, 0.0700, 0.1400, 0.0700, 0.1100],
        [0.1100, 0.2100, 0.0800, 0.0300, 0.2300, 0.2000, 0.0600, 0.1000],
        [0.1300, 0.1100, 0.0100, 0.0700, 0.2500, 0.0900, 0.2000, 0.1400],
        [0.1000, 0.1000, 0.1500, 0.2300, 0.1400, 0.1000, 0.1100, 0.0700],
        [0.1500, 0.1100, 0.0700, 0.0700, 0.1300, 0.2700, 0.0400, 0.1500],
        [0.1300, 0.0900, 0.2200, 0.0800, 0.1400, 0.1300, 0.1100, 0.1000],
        [0.1100, 0.1100, 0.2400, 0.0800, 0.1000, 0.0600, 0.1900, 0.1100],
        [0.0800, 0.2600, 0.0800, 0.0900, 0.2300, 0.0900, 0.1000, 0.0600]],
       grad_fn=<RoundBackward1>)


## Reference check against the course output
Shapes `(4, 8, 16)`, `(4, 8, 8)`, `(4, 8, 16)` and `True` match the course. Below I paste the course's printed matrix
and compare it with mine — on this PyTorch version they are identical.

In [8]:
course_wei0 = torch.tensor([
    [0.19, 0.12, 0.08, 0.23, 0.07, 0.14, 0.07, 0.11],
    [0.11, 0.21, 0.08, 0.03, 0.23, 0.20, 0.06, 0.10],
    [0.13, 0.11, 0.01, 0.07, 0.25, 0.09, 0.20, 0.14],
    [0.10, 0.10, 0.15, 0.23, 0.14, 0.10, 0.11, 0.07],
    [0.15, 0.11, 0.07, 0.07, 0.13, 0.27, 0.04, 0.15],
    [0.13, 0.09, 0.22, 0.08, 0.14, 0.13, 0.11, 0.10],
    [0.11, 0.11, 0.24, 0.08, 0.10, 0.06, 0.19, 0.11],
    [0.08, 0.26, 0.08, 0.09, 0.23, 0.09, 0.10, 0.06]])
mine = wei[0].detach().round(decimals=2)
print("torch version         :", torch.__version__)
print("max |mine - course|   :", (mine - course_wei0).abs().max().item())
print("matches course exactly:", torch.allclose(mine, course_wei0, atol=1e-6))
print("upper triangle non-zero (tokens can see the future):", bool((torch.triu(wei[0], 1) > 0).any()))

torch version         : 2.15.0+cpu
max |mine - course|   : 0.0
matches course exactly: True
upper triangle non-zero (tokens can see the future): True


The last line is the problem lesson 5 fixes: token 3 has non-zero weight on token 4, i.e. it can see the very
character it's supposed to predict.

## Test yourself

**1. What do the query, key and value of a token each represent?**
The query is what the token is searching for, the key is what it advertises about itself, and the value is the
information it passes on. Query·key decides *how much* to attend; the value is *what* gets mixed in.

**2. Why divide the scores by √head_size?**
Dot products of random vectors grow with their length, so without scaling the scores get large and softmax becomes
nearly one-hot — one token takes all the weight and gradients vanish. Scaling keeps the variance around 1.

**3. What shape is `wei` and what does row i mean?**
`(B, T, T)` = (4, 8, 8). Row i of one sequence is how token i splits its attention over all 8 positions; it sums to 1.

## My notes
- Attention is just a learned, data-dependent weighted average: dot products pick the weights, softmax normalises them.
- Surprising: nothing here stops a token attending to later positions — the matrix is full, so it could cheat.